# Ứng dụng dự đoán điểm thi 
> **Cảnh báo:** Ứng dụng và mô hình sử dụng dữ liệu tổng hợp cho mục đích học tập/giảng dạy, không sử dụng để đánh giá sinh viên thực tế.

In [ ]:
from pathlib import Path
import joblib
import numpy as np 
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

MODEL_PATH = Path("artifacts/linear_regression.joblib")
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

artifact = joblib.load(MODEL_PATH)
model = artifact["model"]
feature_names = artifact["feature_names"]

In [ ]:
from IPython.display import HTML, display
import ipywidgets as widgets
import joblib
import numpy as np
import pandas as pd

custom_css = """
<style>
    :root, body, #rendered_cells, .jp-Notebook, div#root {
        --jp-layout-color0: #cc1fee !important;
        --jp-layout-color1: #d53fb2 !important;
        background-color: #f9a6e1 !important;
        font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif !important;
    }
    
    h1, .rendered_html blockquote {
        max-width: 620px;
        margin-left: auto !important;
        margin-right: auto !important;
    }

    .widget-box {
        background-color: #ef78d3 !important;
        padding: 30px 36px !important;
        border-radius: 16px !important;
        box-shadow: 0 10px 25px -5px rgba(0, 0, 0, 0.08), 0 8px 10px -6px rgba(0, 0, 0, 0.04) !important;
        border: 1px solid #d11862 !important;
        max-width: 620px !important;
        margin: 20px auto !important; /* Căn giữa màn hình */
        display: flex !important;
    }


    .widget-box .jupyter-widgets.widget-button {
        background-color: #fa91bb !important;
        color: #070707 !important;
        font-weight: 600 !important;
        border-radius: 8px !important;
        border: none !important;
        height: 38px !important;
        margin-top: 10px !important;
        transition: background-color 0.2s ease !important;
    }
    .widget-box .jupyter-widgets.widget-button:hover {
        background-color: #d01ba6 !important;
    }

    .output pre {
        font-size: 17px !important;
        font-weight: 700 !important;
        color: #0f172a !important;
        background-color: #f8fafc !important;
        padding: 12px !important;
        border-radius: 8px !important;
        border-left: 4px solid #ef3d78 !important;
        margin-top: 14px !important;
    }
</style>
"""
display(HTML(custom_css))

# Code widget giữ nguyên
hours = widgets.FloatSlider(
    value=6.0,
    min=0.0,
    max=20.0,
    step=0.5,
    description="Giờ học:",
    continuous_update=False,
)
attendance = widgets.FloatSlider(
    value=80.0,
    min=50.0,
    max=100.0,
    step=1.0,
    description="Chuyên cần:",
    continuous_update=False,
)
assignments = widgets.IntSlider(
    value=7,
    min=0,
    max=10,
    step=1,
    description="Bài tập:",
    continuous_update=False,
)
previous = widgets.FloatSlider(
    value=6.5,
    min=0.0,
    max=10.0,
    step=0.1,
    description="Điểm trước:",
    continuous_update=False,
)

predict_button = widgets.Button(description="Dự đoán", button_style="primary")
output = widgets.Output()


def input_frame():
  values = {
      "hours_studied": hours.value,
      "attendance": attendance.value,
      "assignments_completed": assignments.value,
      "previous_score": previous.value,
  }
  return pd.DataFrame(
      [[values[name] for name in feature_names]], columns=feature_names
  )


def on_predict(_):
  predict_button.disabled = True
  with output:
    output.clear_output()
    try:
      raw = float(model.predict(input_frame())[0])
      pred_clamped = np.clip(raw, 0, 10)
      print(f"Điểm thi dự đoán: {pred_clamped:.2f} / 10")
    except Exception:
      print("Không thể dự đoán. Hãy kiểm tra dữ liệu và artifact.")
    finally:
      predict_button.disabled = False


predict_button.on_click(on_predict)

container = widgets.VBox(
    [hours, attendance, assignments, previous, predict_button, output]
)
container.add_class("widget-box")
display(container)